# Loan Approval - Exploratory Data Analysis

This notebook is the *exploration* half of the project; everything it settles is
implemented in `src/` and executed by the DVC pipeline. Nothing here is imported
by the pipeline, the API or the tests.

Questions it answers:

1. What does the raw applicant population look like?
2. Which columns carry missing values, and how should they be filled?
3. Which features actually separate approved from rejected applications?
4. Do the engineered affordability features add signal over the raw columns?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.ingestion.ingest import encode_target
from src.transformation.features import ENGINEERED_FEATURES, add_engineered_features
from src.validation.schema import CATEGORICAL_FEATURES, NUMERIC_FEATURES

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

config = load_config()
raw = pd.read_csv(config.get_path("data.raw_path"))
raw.shape

## 1. The raw feed

The DVC-tracked file `data/raw/loan_applications.csv` is the single source of truth.

In [ ]:
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe().T

## 2. Missing values

These fractions decide the imputation strategy encoded in `build_preprocessor()`.

In [ ]:
missing = (raw.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

ax = missing.plot(kind="barh", figsize=(7, 4), color="#4c72b0")
ax.set_xlabel("% missing")
ax.set_title("Missing values by column")
plt.tight_layout()
missing.round(2)

Numeric gaps are filled with the **median** (incomes and loan amounts are
right-skewed, so the mean would be pulled upward) and categorical gaps with the
**most frequent** value. Both are fitted on the training split only.

## 3. Target balance

In [ ]:
frame = raw.copy()
frame["Loan_Status"] = encode_target(frame["Loan_Status"])

balance = frame["Loan_Status"].value_counts(normalize=True).rename({1: "Approved", 0: "Rejected"})
balance.plot(kind="bar", color=["#55a868", "#c44e52"], figsize=(4, 3), title="Target balance")
plt.tight_layout()
balance.round(3)

Roughly two-thirds approvals. That imbalance is mild enough for plain
accuracy to be misleading but not enough to need resampling, which is why the
pipeline selects on **ROC-AUC** and the quality gate also checks **F1**.

## 4. Numeric features vs the decision

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, column in zip(axes.ravel(), NUMERIC_FEATURES):
    for label, colour in ((1, "#55a868"), (0, "#c44e52")):
        values = frame.loc[frame["Loan_Status"] == label, column].dropna()
        ax.hist(values, bins=30, alpha=0.55, color=colour,
                label="Approved" if label else "Rejected", density=True)
    ax.set_title(column)
    ax.legend(fontsize=8)

plt.tight_layout()

`Credit_History` separates the two classes far more sharply than income does -
the single strongest predictor, and the one the API tests assert on.

In [ ]:
frame.groupby("Credit_History")["Loan_Status"].agg(["mean", "count"]).round(3)

## 5. Categorical features vs the decision

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, column in zip(axes.ravel(), CATEGORICAL_FEATURES):
    rates = frame.groupby(column)["Loan_Status"].mean().sort_values()
    rates.plot(kind="bar", ax=ax, color="#4c72b0")
    ax.axhline(frame["Loan_Status"].mean(), color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Approval rate by {column}")
    ax.set_ylabel("approval rate")

plt.tight_layout()

Graduates and semi-urban properties sit above the base rate; rural
properties and the self-employed sit below it. All of these are one-hot encoded
rather than ordinal-encoded, since none of them has a natural order.

## 6. Do the engineered features help?

The features defined in `src/transformation/features.py`:

In [ ]:
enriched = add_engineered_features(frame)
enriched[ENGINEERED_FEATURES].describe().T.round(3)

In [ ]:
correlations = (
    enriched[NUMERIC_FEATURES + ENGINEERED_FEATURES + ["Loan_Status"]]
    .corr(numeric_only=True)["Loan_Status"]
    .drop("Loan_Status")
    .sort_values()
)

ax = correlations.plot(kind="barh", figsize=(7, 5), color="#4c72b0")
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Correlation with approval")
plt.tight_layout()
correlations.round(3)

`DebtToIncomeRatio` correlates more strongly with the outcome than either
`LoanAmount` or `ApplicantIncome` alone - the instalment burden matters more
than the raw numbers, which is why the ratio is computed as a feature rather
than left for the model to infer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, colour, name in ((1, "#55a868", "Approved"), (0, "#c44e52", "Rejected")):
    subset = enriched[enriched["Loan_Status"] == label]
    axes[0].scatter(subset["TotalIncome"], subset["LoanAmount"], s=8, alpha=0.4,
                    color=colour, label=name)
    axes[1].hist(subset["DebtToIncomeRatio"].clip(upper=2), bins=40, alpha=0.55,
                 color=colour, label=name, density=True)

axes[0].set(xscale="log", xlabel="Total income (log)", ylabel="Loan amount (thousands)",
            title="Income vs requested amount")
axes[1].set(xlabel="Debt-to-income ratio", title="Affordability by outcome")
for ax in axes:
    ax.legend(fontsize=8)

plt.tight_layout()

## 7. What this means for the pipeline

| Finding | Where it is implemented |
| --- | --- |
| Skewed incomes, missing amounts | median imputation + `StandardScaler` in `build_preprocessor()` |
| Unordered categories, unseen values possible at serving time | `OneHotEncoder(handle_unknown="ignore")` |
| Credit history dominates | asserted in `tests/test_training.py::test_credit_history_drives_the_decision` |
| Affordability beats raw amounts | `DebtToIncomeRatio`, `LoanAmountPerMonth` in `ENGINEERED_FEATURES` |
| Mild class imbalance | model selection on ROC-AUC, release gate on ROC-AUC **and** F1 |
| Distributions worth watching in production | PSI features in `src/monitoring/drift.py` |

Reproduce everything below this notebook with:

```bash
dvc repro
```
